In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from rich.console import Console  # only for fancy text formatting
from anthropic import Anthropic

load_dotenv(override=True)
console = Console(force_jupyter=False)

client = Anthropic()
MODEL = "claude-haiku-4-5"
MAX_TOKENS = 1024

In [6]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": MODEL,
        "max_tokens": MAX_TOKENS,
        "messages": messages,
        "stop_sequences": stop_sequences,
        # this will work with older (<1.1.0) SDK
        # "temperature": temperature,
        # ------------------------------
        # for 1.1.0+ SDK use the following
        "extra_body": {"temperature": temperature},
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [7]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used
        to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related 
        tasks. Generate an array of JSON objects, each representing task that requires Python, 
        JSON, or a Regex to complete. 

        Example output:
        ```json
        [
            {
                "task": "Description of task",
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single 
          JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences=["```"])
    return json.loads(response)

In [ ]:
# let's test the function defined above

dataset = generate_dataset()
console.print(dataset)

# write the dataset to JSON file
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[
    {
        'task': 'Write a regular expression to validate an AWS S3 bucket name. 
S3 bucket names must be between 3-63 characters, contain only lowercase 
letters, numbers, hyphens, and periods, start and end with a letter or number, 
and cannot contain consecutive hyphens or periods.'
    },
    {
        'task': 'Write a Python function that takes an AWS CloudFormation 
template (as a dictionary) and returns a list of all resource logical IDs that 
have a DependsOn attribute.'
    },
    {
        'task': "Write a JSON object that represents an AWS IAM policy 
statement allowing read-only access to a specific S3 bucket named 
'my-data-bucket' including ListBucket, GetObject, and GetObjectVersion 
actions."
    }
]
